# Stage 3 Clinical NLP: Train-Only Data Augmentation Demo
This interactive notebook demonstrates the clinically safe, entity-preserving data augmentation pipeline for precision oncology.

In [ ]:
import sys
from pathlib import Path
import json
import random
import pandas as pd

# Set up paths
SRC_DIR = Path('../src')
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from entity_preserving_augmentation import augment_document_preserving_entities, verify_entity_spans
from terminology_augmentation import apply_terminology_augmentation
from sentence_augmentation import permute_vitals_clauses, permute_lab_chemistry_clauses
from duplicate_detection import compute_jaccard_similarity
from quality_control import QualityControlGate

print('All modules loaded successfully!')

## 1. Inspect Sample Training Clinical Document

In [ ]:
sample_text = (
    "ONCOLOGY CONSULTATION PROGRESS NOTE\n"
    "Patient ID: PT-000002\n"
    "Demographics: 66-year-old male presenting for Cycle 1 evaluation.\n\n"
    "DIAGNOSIS & MOLECULAR PROFILING:\n"
    "Primary Diagnosis: Stage III Unknown.\n"
    "Genomic Profile: Confirmed driver mutation in TP53. Secondary mutation: None/Unknown.\n\n"
    "CLINICAL ASSESSMENT & LABORATORY REVIEW:\n"
    "Vitals: Blood pressure 134/68 mmHg, Heart rate 67 bpm, SpO2 98% on room air.\n"
    "Physical examination reveals no jaundice, no acute respiratory distress, and clear lung fields bilaterally.\n\n"
    "TREATMENT PLAN & REGIMEN:\n"
    "Administer scheduled therapy with carboplatin at a dosage of 450.0 mg.\n"
    "Pre-medications ordered per protocol."
)

sample_entities = [
    {"start": sample_text.find("TP53"), "end": sample_text.find("TP53") + len("TP53"), "label": "GENE_MUTATION", "text": "TP53"},
    {"start": sample_text.find("carboplatin"), "end": sample_text.find("carboplatin") + len("carboplatin"), "label": "DRUG_NAME", "text": "carboplatin"},
    {"start": sample_text.find("450.0 mg"), "end": sample_text.find("450.0 mg") + len("450.0 mg"), "label": "DOSAGE", "text": "450.0 mg"}
]

is_valid, msg = verify_entity_spans(sample_text, sample_entities)
print(f'Original Entity Span Invariant: {is_valid} ({msg})')

## 2. Execute Entity-Preserving Clinical Augmentation

In [ ]:
rng = random.Random(42)
aug_text, aug_ents, success, desc = augment_document_preserving_entities(
    sample_text, sample_entities, 'oncology_consultation', rng, strategy='composite'
)

print(f'Augmentation Success: {success} | Method: {desc}\n')
print('=== AUGMENTED TEXT ===')
print(aug_text)
print('\n=== RE-ALIGNED ENTITY SPANS ===')
for e in aug_ents:
    print(f"{e['label']}: '{aug_text[e['start']:e['end']]}' (span {e['start']}:{e['end']})")

## 3. Quality Control Verification and Jaccard Similarity

In [ ]:
jaccard = compute_jaccard_similarity(sample_text, aug_text, n=2)
print(f'2-gram Jaccard Similarity: {jaccard:.4f} (Safe bounds: [0.50, 0.98])')

qc = QualityControlGate()
source_row = {
    'document_id': 'DOC-000004',
    'patient_id': 'PT-000002',
    'text': sample_text,
    'urgency_level': 'HIGH',
    'hazard_type': 'NONE'
}
passed, reason = qc.evaluate(aug_text, aug_ents, source_row, desc, {'PT-000002'})
print(f'10-Point QC Gate Verdict: {passed} ({reason})')